# Regular versus fast spiking, on the newly filtered units

Repeats the RS/FS assessment on the units surviving the new filtering.

**Why the firing metrics are recomputed here rather than loaded.** The existing
`sua_firing_NATIM` dataframes were computed on the earlier good-units files. The
new filtering starts from every sorted unit, so it admits units the earlier
selection rejected, and those have no entry there. Joining the old metrics would
silently restrict the analysis to units the earlier pipeline happened to like,
which is the opposite of what a refiltering is for. The metrics are therefore
computed from the filtered spike times, using the same functions as the original
script.

**What the earlier attempt concluded.** The evidence conflicted rather than
being merely weak. Narrow Biphasic was more regular and less bursty than Wide,
consistent with fast spiking, but fired more slowly, responded later and with
more jitter, all of which point the other way. The full phenotype, regular and
non-bursty and fast together, appeared in 0.1 to 0.2 percent of Narrow Biphasic
against 4.5 percent of Narrow Triphasic. Classifier accuracy fell after
cleaning, and the rate contrast changed sign between filtering levels.

The question here is whether a cleaner unit set changes any of that.

In [ ]:
import os
os.chdir('/CSNG/studekat/ripple_paper_clean_copy/code_new_filter')

In [ ]:
from functions_analysis import *
from functions_firing import *
import pandas as pd, numpy as np, yaml, pickle, neo
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.stats import zscore
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score, StratifiedKFold
from statsmodels.stats.multitest import multipletests
import warnings
warnings.simplefilter(action='ignore', category=pd.errors.SettingWithCopyWarning)

with open("/CSNG/studekat/ripple_paper_clean_copy/code_new_filter/params_analysis.yml") as f:
    P = yaml.safe_load(f)

DF_FOLDER = '/CSNG/studekat/ripple_paper_clean_copy/dataframes_new_filter'
TYPE_REC = 'NATIM'
MONKEYS  = ['N','F']
DATA_FOLDER = P['natim_data_folder']
AREA = 'V12'

FINAL_CLASSES   = P['final_classes']
CLASS_COLORS    = P['colors_class']
WIDTH_INTERVALS = P['width_intervals']
PEAK_HEIGHT     = P['first_peak_height']
CLASS_DICT = {'DOWN_narrow_shallow':'NarrBI','DOWN_narrow_sharp':'NarrTRI',
              'DOWN_wide':'Wide','DOWN_medium_shallow':'MedBI',
              'DOWN_medium_sharp':'MedTRI','UP':'Pos'}
KEY = ['monkey','date','array','cell_name']

FS_CAND, RS_CAND = 'DOWN_narrow_shallow', 'DOWN_wide'
classes_no_up = [c for c in FINAL_CLASSES if c != 'UP']

## 1. Paths and epochs

Epoch definitions match the original firing script exactly, so the metrics are
comparable. In this paradigm the median stimulus onset asynchrony is about
417 ms, so there is no quiescent pre-stimulus period within a trial and the
baseline is built from the longer inter-trial gaps.

In [ ]:
def recording_folder(monkey, date, type_rec, data_folder):
    if type_rec == 'NATIM':
        return f'{data_folder}/macaque{monkey}_TVSD_{date}'
    return f'{data_folder}/macaque{monkey}_{type_rec}_{date}'

def file_stem(monkey, date, type_rec):
    if type_rec == 'NATIM':
        return f'macaque{monkey}_TVSD_{date}'
    return f'macaque{monkey}_{type_rec}_{date}'

def spike_path(monkey, date, array, type_rec, data_folder, kind='filtered'):
    base = recording_folder(monkey, date, type_rec, data_folder)
    stem = file_stem(monkey, date, type_rec)
    sub, suf = {'all':      ('spikes_KS4',      'spikes_KS4_unfiltered'),
                'good':     ('spikes_KS4',      'spikes_KS4'),
                'filtered': ('spikes_filtered', 'spikes_KS4_filtered')}[kind]
    p = f'{base}/{sub}/{stem}_Array{array}_{suf}.nix'
    return p if os.path.isfile(p) else None


def load_trial_metadata(monkey, date):
    folder = recording_folder(monkey, date, 'NATIM', DATA_FOLDER)
    stem = file_stem(monkey, date, 'NATIM')
    for p in [f'{folder}/{stem}_trial_metadata.csv',
              f'{folder}/metadata/{stem}_trial_metadata.csv',
              f'{folder}/trial_metadata.csv']:
        if os.path.isfile(p):
            return pd.read_csv(p)
    return None


CFG = {
    'min_gap_s': 1.0, 'trim_front_s': 0.4, 'trim_back_s': 0.1,
    'transient_s': (0.03, 0.10), 'late_s': (0.10, 0.25), 'pre_s': (-0.10, 0.0),
    'psth_pre_s': 0.10, 'psth_post_s': 0.35, 'psth_bin_s': 0.005,
    'psth_smooth_bins': 3,
    'lat_n_sd': 3.0, 'lat_n_consec': 2, 'lat_search_to_s': 0.25,
    'fsl_min_s': 0.02, 'fsl_max_s': 0.25,
    'burst_th_s': 0.008,
    'adapt_early_s': (0.03, 0.08), 'adapt_late_s': (0.12, 0.25),
    'min_spikes_baseline': 100, 'min_spikes_evoked': 100, 'min_trials': 200,
}

## 2. Compute the firing metrics from the filtered files

The slow step. Set `MAX_RECORDINGS` for a first pass.

In [ ]:
MAX_RECORDINGS = 3

def annot(st, key, default=np.nan):
    v = st.annotations.get(key, default)
    if isinstance(v, str) and v.strip().lower() == 'nan':
        return np.nan
    return v


rows, n_rec = [], 0
for monkey in MONKEYS:
    for date in P['dates'][monkey][TYPE_REC]:
        if MAX_RECORDINGS is not None and n_rec >= MAX_RECORDINGS:
            break
        trials = load_trial_metadata(monkey, date)
        if trials is None:
            print(f'   {monkey} {date}: no trial metadata, skipped'); continue
        if 'Success' in trials.columns:
            trials = trials[trials['Success'] == 1]
        onsets_all = np.sort(trials['Trial_start_s'].values.astype(float))

        found = False
        for array in range(1, 17):
            p = spike_path(monkey, date, array, TYPE_REC, DATA_FOLDER, 'filtered')
            if p is None:
                continue
            found = True
            try:
                io = neo.NixIO(p, 'ro'); blk = io.read_block()
                sts = blk.segments[0].spiketrains
                if not len(sts):
                    io.close(); continue
                t_start = float(sts[0].t_start.rescale('s').magnitude)
                t_stop  = float(sts[0].t_stop.rescale('s').magnitude)
                onsets = onsets_all[(onsets_all >= t_start) &
                                    (onsets_all <= t_stop - CFG['psth_post_s'])]
                if len(onsets) < CFG['min_trials']:
                    io.close(); continue

                base_win  = build_baseline_windows(
                    onsets, t_start, t_stop, min_gap_s=CFG['min_gap_s'],
                    trim_front_s=CFG['trim_front_s'], trim_back_s=CFG['trim_back_s'])
                trans_win = build_trial_windows(onsets, *CFG['transient_s'])
                late_win  = build_trial_windows(onsets, *CFG['late_s'])
                base_dur  = total_window_duration(base_win)

                for st in sts:
                    t = np.sort(np.asarray(st.times.rescale('s').magnitude,
                                           dtype=float))
                    d = {'monkey': monkey, 'date': date, 'array': array,
                         'cell_name': annot(st, 'nix_name', ''),
                         'n_spikes': len(t), 'n_trials_used': len(onsets)}
                    w = annot(st, 'avg_wf', None)
                    wz = annot(st, 'avg_wf_zscored', None)
                    d['avg_wf'] = np.asarray(w, float) if w is not None and not isinstance(w, str) else None
                    d['avg_wf_zscored'] = np.asarray(wz, float) if wz is not None and not isinstance(wz, str) else None
                    d['Area'] = annot(st, 'Area', 'unknown')
                    d['Electrode_ID'] = annot(st, 'Electrode_ID')

                    sb = spikes_in_windows(t, base_win)
                    stz = spikes_in_windows(t, trans_win)
                    sl = spikes_in_windows(t, late_win)
                    d['n_spikes_baseline'] = int(sum(len(s) for s in sb))
                    d['n_spikes_evoked'] = int(sum(len(s) for s in stz) +
                                               sum(len(s) for s in sl))

                    d['FR_baseline']  = firing_rate(sb, base_win)
                    d['FR_transient'] = firing_rate(stz, trans_win)
                    d['FR_late']      = firing_rate(sl, late_win)

                    rate, centres = psth(t, onsets, t_pre_s=CFG['psth_pre_s'],
                                         t_post_s=CFG['psth_post_s'],
                                         bin_s=CFG['psth_bin_s'])
                    pk, pk_t = peak_evoked_rate(rate, centres, 0.0,
                                                CFG['lat_search_to_s'],
                                                smooth_bins=CFG['psth_smooth_bins'])
                    d['FR_peak_evoked'] = pk
                    d['modulation_index'] = modulation_index(d['FR_transient'],
                                                             d['FR_baseline'])

                    if base_dur > 0 and d['n_spikes_baseline'] > 0:
                        bc = []
                        for (w0, w1), seg in zip(base_win, sb):
                            nb = int((w1 - w0)/CFG['psth_bin_s'])
                            if nb < 1: continue
                            e = np.linspace(w0, w0 + nb*CFG['psth_bin_s'], nb+1)
                            bc.append(np.histogram(seg, bins=e)[0])
                        sd = float(np.std(np.concatenate(bc)/CFG['psth_bin_s'])) \
                             if bc else np.nan
                    else:
                        sd = np.nan
                    d['latency_psth_s'] = response_latency(
                        rate, centres, d['FR_baseline'], sd,
                        n_sd=CFG['lat_n_sd'], n_consecutive=CFG['lat_n_consec'],
                        search_to_s=CFG['lat_search_to_s'])
                    fsl, fslsd, n_fsl = first_spike_latency_stats(
                        t, onsets, t_min_s=CFG['fsl_min_s'], t_max_s=CFG['fsl_max_s'])
                    d['first_spike_latency_s'] = fsl
                    d['first_spike_jitter_s'] = fslsd

                    isi_b = isis_within_segments(sb)
                    isi_e = isis_within_segments(stz + sl)
                    d['CV_ISI_evoked'] = cv_isi(isi_e)
                    d['CV2_evoked'] = cv2(stz + sl)
                    d['LV_evoked'] = local_variation(stz + sl)
                    d['LV_baseline'] = local_variation(sb)
                    d['burst_index_evoked'] = burst_index(isi_e, th_s=CFG['burst_th_s'])
                    fb, spb = burst_stats(stz + sl, th_s=CFG['burst_th_s'])
                    d['frac_spikes_in_burst_evoked'] = fb
                    d['spikes_per_burst_evoked'] = spb
                    d['psth_decay_ratio'] = adaptation_ratio(
                        rate, centres, early=CFG['adapt_early_s'],
                        late=CFG['adapt_late_s'])

                    d['enough_evoked'] = bool(
                        d['n_spikes_baseline'] >= CFG['min_spikes_baseline'] and
                        d['n_spikes_evoked'] >= CFG['min_spikes_evoked'])
                    rows.append(d)
                io.close(); del blk
            except Exception as e:
                print(f'   {monkey} {date} array {array}: {e}')
        if found:
            n_rec += 1
            print(f'   {monkey} {date}: {len(rows)} units so far', flush=True)

df = pd.DataFrame(rows)
print(f'\n{len(df)} filtered units with firing metrics, from {n_rec} recordings')
print(f'{int(df["enough_evoked"].sum())} have enough spikes in both windows')

## 3. Classify, and restrict to V1/V2

In [ ]:
d = df[df['avg_wf'].notna()].copy()
if d['avg_wf_zscored'].isna().any():
    m = d['avg_wf_zscored'].isna()
    d.loc[m, 'avg_wf_zscored'] = [zscore(w) for w in d.loc[m, 'avg_wf']]
d = aux_add_waveform_prop(d)
d['amp_wf_zscored'] = [np.max(w) - np.min(w) for w in d['avg_wf_zscored']]
d = aux_add_width_classes(d, width_intervals=WIDTH_INTERVALS)
d = aux_add_up_down_classes(d)
d = aux_add_final_classes(d, final_classes=FINAL_CLASSES,
                          peak_height_th=PEAK_HEIGHT)
d['area_merged'] = [a if a in ('V4','IT') else 'V12' for a in d['Area'].astype(str)]

print(f'{len(d)} classified')
d = d[(d['area_merged'] == AREA) & d['enough_evoked']]
print(f'{len(d)} in {AREA} with enough spikes')
print()
print(d['final_class'].value_counts().rename(index=CLASS_DICT).to_string())

METRICS = [m for m in ['FR_baseline','FR_transient','FR_peak_evoked',
                       'modulation_index','LV_evoked','LV_baseline','CV2_evoked',
                       'CV_ISI_evoked','burst_index_evoked',
                       'frac_spikes_in_burst_evoked','spikes_per_burst_evoked',
                       'psth_decay_ratio','first_spike_latency_s',
                       'first_spike_jitter_s'] if m in d.columns]

def rank_biserial(x, y):
    x, y = np.asarray(x), np.asarray(y)
    if len(x) == 0 or len(y) == 0: return np.nan
    u = stats.mannwhitneyu(x, y, alternative='two-sided').statistic
    return 2*u/(len(x)*len(y)) - 1

## 4. The rate contradiction

Fast-spiking interneurons fire faster than pyramidal cells. A positive effect
means Narrow Biphasic is faster than Wide, which is what the hypothesis
predicts. The earlier analysis found the opposite.

In [ ]:
rows = []
a_all = d[d['final_class'] == FS_CAND]
b_all = d[d['final_class'] == RS_CAND]
for m in ['FR_baseline','FR_transient','FR_peak_evoked']:
    a, b = a_all[m].dropna(), b_all[m].dropna()
    if len(a) < 10 or len(b) < 10: continue
    rows.append({'metric': m, 'n_NarrBI': len(a), 'n_Wide': len(b),
                 'median_NarrBI': round(np.median(a), 2),
                 'median_Wide': round(np.median(b), 2),
                 'rank_biserial': round(rank_biserial(a, b), 3),
                 'p': f'{stats.mannwhitneyu(a, b).pvalue:.1e}'})
df_rate = pd.DataFrame(rows)
print('Positive rank-biserial means Narrow Biphasic is FASTER, the')
print('fast-spiking direction.')
print()
print(df_rate.to_string(index=False))
print()
print('Earlier result on the old unit set, for comparison:')
print('   FR_baseline -0.015 unfiltered, -0.330 at the strict level')

## 5. Effect sizes across all metrics

For regularity and bursting a **negative** effect is the fast-spiking direction,
since Narrow Biphasic should be more regular and less bursty.

In [ ]:
rows = []
for m in METRICS:
    a, b = a_all[m].dropna(), b_all[m].dropna()
    if len(a) < 10 or len(b) < 10: continue
    u = stats.mannwhitneyu(a, b)
    rows.append({'metric': m,
                 'median_NarrBI': round(np.median(a), 4),
                 'median_Wide': round(np.median(b), 4),
                 'rank_biserial': round(rank_biserial(a, b), 3),
                 'p': u.pvalue, 'n_NarrBI': len(a), 'n_Wide': len(b)})
df_eff = pd.DataFrame(rows)
df_eff['p_holm'] = multipletests(df_eff['p'], method='holm')[1]
df_eff['p'] = df_eff['p'].apply(lambda x: f'{x:.1e}')
df_eff = df_eff.reindex(df_eff['rank_biserial'].abs()
                        .sort_values(ascending=False).index)
print(df_eff.to_string(index=False))

fig, ax = plt.subplots(figsize=(7.5, 5), dpi=120)
cols = ['maroon' if s else 'lightgray' for s in (df_eff['p_holm'] < 0.05)]
ax.barh(df_eff['metric'], df_eff['rank_biserial'], color=cols, alpha=0.9)
ax.axvline(0, color='k', lw=1)
ax.set_xlabel('rank-biserial, Narrow Biphasic vs Wide')
ax.set_title('Filled bars survive Holm correction', fontsize=10)
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
plt.tight_layout(); plt.show()

In [ ]:
show = df_eff['metric'].tolist()[:6]
fig, axes = plt.subplots(2, 3, figsize=(14, 7), dpi=115)
for ax, m in zip(axes.flat, show):
    sub = d[d['final_class'].isin(classes_no_up)][['final_class', m]].dropna()
    sns.violinplot(data=sub, x='final_class', y=m, hue='final_class',
                   order=classes_no_up, palette=CLASS_COLORS, inner='box',
                   cut=0, ax=ax, legend=False, linewidth=1.1)
    for v in ax.collections: v.set_alpha(0.7)
    ax.set_xticks(range(len(classes_no_up)))
    ax.set_xticklabels([CLASS_DICT[c] for c in classes_no_up], fontsize=7,
                       rotation=30)
    ax.set_xlabel(''); ax.set_ylabel(m, fontsize=9)
    if m.startswith('FR_'): ax.set_yscale('log')
    ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
plt.tight_layout(); plt.show()

## 6. Is there a coherent fast-spiking phenotype?

The strongest evidence would be units that are simultaneously regular,
non-bursty and fast. Defined by within-set terciles so it does not depend on
absolute thresholds. The earlier analysis found this phenotype in 0.1 to 0.2
percent of Narrow Biphasic against 4.5 percent of Narrow Triphasic.

In [ ]:
need = ['LV_evoked','burst_index_evoked','FR_baseline']
dd = d[need + ['final_class']].dropna()
lo_lv    = dd['LV_evoked']          <= dd['LV_evoked'].quantile(1/3)
lo_burst = dd['burst_index_evoked'] <= dd['burst_index_evoked'].quantile(1/3)
hi_fr    = dd['FR_baseline']        >= dd['FR_baseline'].quantile(2/3)
dd['is_FS'] = lo_lv & lo_burst & hi_fr

rows = []
for cl in FINAL_CLASSES:
    sub = dd[dd['final_class'] == cl]
    if not len(sub): continue
    rows.append({'class': CLASS_DICT[cl], 'n': len(sub),
                 'n_FS_phenotype': int(sub['is_FS'].sum()),
                 'pct': round(100*sub['is_FS'].mean(), 2)})
print(pd.DataFrame(rows).to_string(index=False))
print()
print(f'overall {100*dd["is_FS"].mean():.2f}% of units meet all three')

a = dd.loc[dd['final_class'] == FS_CAND, 'is_FS']
b = dd.loc[dd['final_class'] == RS_CAND, 'is_FS']
if len(a) >= 10 and len(b) >= 10 and (a.sum() + b.sum()) > 0:
    ct = np.array([[a.sum(), len(a)-a.sum()], [b.sum(), len(b)-b.sum()]])
    odds, pv = stats.fisher_exact(ct)
    print(f'NarrBI vs Wide: odds ratio {odds:.2f}, p {pv:.2e}')
    print('   above 1 means the phenotype is enriched in Narrow Biphasic')

## 7. Is LV bimodal within Narrow Biphasic?

The earlier analysis found two components, and the low-LV one turned out to be
slow rather than fast, which is why the fast-spiking reading of it failed. Tested
properly here, since a mixture will split a skewed distribution regardless.

In [ ]:
try:
    import diptest
    HAVE_DIP = True
except ImportError:
    HAVE_DIP = False
    print('diptest not installed; install with  pip install diptest')

for cl in [FS_CAND, RS_CAND]:
    v = d.loc[d['final_class'] == cl, 'LV_evoked'].dropna().values
    if len(v) < 150:
        print(f'{CLASS_DICT[cl]}: n={len(v)}, too few'); continue
    dp = diptest.diptest(v)[1] if HAVE_DIP else np.nan
    x = v.reshape(-1, 1)
    bics = [GaussianMixture(k, random_state=0, n_init=3).fit(x).bic(x)
            for k in [1, 2, 3]]
    print(f'{CLASS_DICT[cl]}: n={len(v)}, median LV {np.median(v):.3f}, '
          f'dip p {dp:.4f}, BIC prefers k={[1,2,3][int(np.argmin(bics))]}')

fig, ax = plt.subplots(figsize=(6, 3.8), dpi=120)
for cl in classes_no_up:
    v = d.loc[d['final_class'] == cl, 'LV_evoked'].dropna()
    if len(v) > 30:
        sns.kdeplot(v, ax=ax, color=CLASS_COLORS[cl], label=CLASS_DICT[cl], lw=2)
ax.set_xlabel('LV (evoked)'); ax.legend(fontsize=8, frameon=False)
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
plt.tight_layout(); plt.show()

## 8. Do the firing metrics predict waveform class?

A single number for how much the two descriptions agree. Chance is the balanced
rate. The earlier analysis gave 0.699 unfiltered, falling to 0.652 after
cleaning.

In [ ]:
MV = [m for m in ['FR_baseline','FR_transient','FR_peak_evoked',
                  'modulation_index','CV2_evoked','LV_evoked',
                  'burst_index_evoked','frac_spikes_in_burst_evoked',
                  'first_spike_latency_s','first_spike_jitter_s']
      if m in d.columns]
dm = d[d['final_class'].isin(classes_no_up)][MV + ['final_class']].dropna()
print(f'{dm.shape[0]} units in the multivariate analysis')

X = dm[MV].values.astype(float)
for i, m in enumerate(MV):
    if m.startswith('FR_'):
        X[:, i] = np.log10(np.clip(X[:, i], 0, None) + 0.1)
X = StandardScaler().fit_transform(X)
y = dm['final_class'].values

clf = RandomForestClassifier(400, random_state=0, class_weight='balanced', n_jobs=-1)
sc = cross_val_score(clf, X, y, cv=StratifiedKFold(5, shuffle=True, random_state=0),
                     scoring='balanced_accuracy')
print(f'balanced accuracy {sc.mean():.3f} +/- {sc.std():.3f}')
print(f'chance             {1/len(np.unique(y)):.3f}')

clf.fit(X, y)
imp = pd.Series(clf.feature_importances_, index=MV).sort_values()
fig, ax = plt.subplots(figsize=(5.5, 4), dpi=120)
imp.plot.barh(ax=ax, color='gray', alpha=0.85)
ax.set_xlabel('feature importance')
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
plt.tight_layout(); plt.show()

## 9. Verdict

In [ ]:
print('='*68)
print('REGULAR VERSUS FAST SPIKING, NEW FILTERED UNITS')
print('='*68)
print(f'\n{len(d)} units in {AREA}, '
      f'{len(a_all)} Narrow Biphasic and {len(b_all)} Wide\n')

FS_EXPECT = {'FR_baseline': +1, 'FR_transient': +1, 'FR_peak_evoked': +1,
             'LV_evoked': -1, 'LV_baseline': -1, 'CV2_evoked': -1,
             'burst_index_evoked': -1, 'frac_spikes_in_burst_evoked': -1,
             'spikes_per_burst_evoked': -1, 'first_spike_latency_s': -1,
             'first_spike_jitter_s': -1, 'psth_decay_ratio': +1}
agree = []
print('%-30s %8s %10s  %s' % ('metric', 'rb', 'FS expects', 'consistent'))
for _, r in df_eff.iterrows():
    m = r['metric']
    if m not in FS_EXPECT: continue
    ok = (r['rank_biserial'] * FS_EXPECT[m]) > 0
    agree.append(ok)
    print('%-30s %+8.3f %10s  %s' % (
        m, r['rank_biserial'],
        'higher' if FS_EXPECT[m] > 0 else 'lower', 'yes' if ok else 'NO'))
print(f'\n{sum(agree)} of {len(agree)} metrics point toward Narrow Biphasic')
print('being the more fast-spiking of the two.')
print()
print('Rate and latency are the most reliable fast-spiking markers across')
print('species; regularity and bursting are a single property seen three ways,')
print('so a count of metrics overstates the independence of the evidence.')
print('='*68)

In [ ]:
OUT = f'{DF_FOLDER}/rs_fs_filtered'
ensure_dir_exists(OUT)
cols = [c for c in d.columns if c not in ('avg_wf','avg_wf_zscored')]
d[cols].to_csv(f'{OUT}/firing_metrics_filtered_{TYPE_REC}.csv', index=False)
d[KEY + METRICS + ['final_class']].to_pickle(
    f'{OUT}/firing_metrics_filtered_{TYPE_REC}.pkl')
df_eff.to_csv(f'{OUT}/effect_sizes_{TYPE_REC}.csv', index=False)
print('saved to', OUT)

## Notes

**The metrics are computed here, not loaded.** The existing dataframes cover only
units the earlier filtering kept, so using them would restrict the analysis to
that selection and hide whatever the new filtering admits. The epoch definitions
and metric functions are the same, so the values remain comparable.

**Read the direction, not the count.** Regularity, CV2, burst index and fraction
of spikes in bursts are four views of one interval distribution, so four metrics
agreeing is one line of evidence rather than four. Firing rate and response
latency are separate and are the more reliable fast-spiking markers.

**The phenotype test in section 6 is the strongest evidence available**, because
it requires the three properties together rather than counting them separately.

**Two caveats that filtering does not remove.** Units recorded across several
blocks within a day still enter every test repeatedly, so effect sizes rather
than p values are the quantity to interpret. And none of this speaks to synaptic
sign: fast spiking is enriched for parvalbumin-positive cells but is not
equivalent to inhibitory identity, and somatostatin and VIP interneurons fire in
the regular-spiking range. Only the monosynaptic labels can settle that.